# 显微血小板图像分析 Pipeline v9
## 架构总览
```
┌────────────────────────────────────────────────────────────────────┐
│  Dual-Branch ResNet18 + DropPath + Layer-SE + time_head            │
│  Phase [B,3,H,W] ──→ Stem → Layer1→SE1 → ... → Layer4→SE4 ─┐     │
│                                                               ├→   │
│  BF    [B,1,H,W] ──→ Stem → Layer1→SE1 → ... → Layer4→SE4 ─┘     │
│         Fusion → GAP → feat[B,512] → Dropout → proj_cls → emb     │
│                                              └→ drug4_head  (★ v9) │
│                                              └→ time_head   (★ v8) │
│                                              └→ proj_supcon        │
└────────────────────────────────────────────────────────────────────┘
```
## v9 主要改动 (相对 v8)
| # | 改动 | 位置 | 原因 |
|---|------|------|------|
| 1 | 删除 Phase A 二分类训练 | Step 1/4/5/6 | 直接四分类，无需层级分解 |
| 2 | 删除 Phase B 三分类训练 | Step 1/4/5/6 | 直接四分类，无需层级分解 |
| 3 | binary_head → drug4_head (4分类) | Step 4 | 直接预测4个药物类别 |
| 4 | 删除 drug3_head | Step 4 | 四分类头已包含所有类别 |
| 5 | Phase AB 合并为单一预热阶段 | Step 6 | drug4_acc≥0.75×2 → Phase C |
| 6 | DirectDrug4Loss 替代 HierarchicalLoss | Step 5 | W_DRUG4×CE + W_TIME×CE_time |
| 7 | drug4_weight=[1,1,1,3] 处理不平衡 | Step 5 | nodrug 样本较少 |
| 8 | Phase C 完整保留 v12 设计 | Step 5b/6 | 时间结构化训练不变 |
| 9 | forward 返回 4 个值 | Step 4 | (drug4, time, emb_cls, emb_con) |


In [ ]:
# -*- coding: utf-8 -*-
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'tifffile', 'umap-learn', 'scikit-learn',
    'pandas', 'tqdm', 'matplotlib', 'seaborn',
    'h5py', 'Pillow', 'scipy',
], check=True)

import torch, torchvision
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

import shutil
files      = ['train.h5', 'val.h5', 'test.h5']
source_dir = '/content/drive/MyDrive/drug_effect/hdf5_2'
target_dir = '/content/data'
os.makedirs(target_dir, exist_ok=True)

print('🚀 开始批量复制文件到会话空间...')
for file_name in files:
    src = os.path.join(source_dir, file_name)
    dst = os.path.join(target_dir, file_name)
    if os.path.exists(src):
        print(f'  📦 复制 {file_name} ...', end='')
        shutil.copy(src, dst)
        print(f' ✅ ({os.path.getsize(dst)/1e9:.2f} GB)')
    else:
        print(f'  ❌ 跳过: {file_name}')
print('✨ 完成！', os.listdir(target_dir))


Mounted at /content/drive
PyTorch     : 2.10.0+cu128
Torchvision : 0.25.0+cu128
CUDA        : True
GPU         : Tesla T4
VRAM        : 15.6 GB
🚀 开始批量复制文件到会话空间...
  📦 复制 train.h5 ... ✅ (4.46 GB)
  📦 复制 val.h5 ... ✅ (1.50 GB)
  📦 复制 test.h5 ... ✅ (1.49 GB)
✨ 完成！ ['val.h5', 'test.h5', 'train.h5']


## Step 1 — 全局配置 ★ v9

In [ ]:
import os, torch

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True

DRIVE_ROOT       = '/content/drive/MyDrive/drug_effect'
LABELS_CSV       = os.path.join(DRIVE_ROOT, 'train/labels.csv')
VAL_LABELS_CSV   = os.path.join(DRIVE_ROOT, 'val/labels_val.csv')
TEST_LABELS_CSV  = os.path.join(DRIVE_ROOT, 'test/labels_test.csv')

PIPE_DIR  = os.path.join(DRIVE_ROOT, 'pipeline_c')
EMB_DIR   = os.path.join(PIPE_DIR, 'embeddings')
ANA_DIR   = os.path.join(PIPE_DIR, 'analysis')
MT_DIR    = os.path.join(PIPE_DIR, 'multitask')
TRAJ_DIR  = os.path.join(PIPE_DIR, 'trajectory')
FINAL_DIR = os.path.join(PIPE_DIR, 'final')

HDF5_TRAIN = '/content/data/train.h5'
HDF5_VAL   = '/content/data/val.h5'
HDF5_TEST  = '/content/data/test.h5'

PHASE_KEYWORD       = '_phase'
BRIGHTFIELD_KEYWORD = '_brightfield'
IMG_EXT             = '.tiff'

# ── 类别定义 ──────────────────────────────────────────────────────────────────
DRUG_CLASSES = ['cangrelor', 'aspirin', 'eptifibatide', 'nodrug']
TIME_CLASSES = ['0min', '5min', '10min', '15min', '20min']
N_DRUG       = len(DRUG_CLASSES)   # 4
N_TIME       = len(TIME_CLASSES)   # 5
DRUG2IDX     = {d: i for i, d in enumerate(DRUG_CLASSES)}
TIME2IDX     = {t: i for i, t in enumerate(TIME_CLASSES)}

# ★ v9: 不再有 BINARY_CLASSES / N_BINARY / DRUG3_CLASSES / N_DRUG3

BACKBONE      = 'resnet18'
PRETRAINED    = True
EMBEDDING_DIM = 256

CROP_SIZE    = 512
BATCH_SIZE   = 96
EPOCHS_MT     = 20   # Phase AB + C 合并总轮数上限
LR_BACKBONE  = 5e-5
LR_HEAD      = 2e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
NUM_WORKERS  = 8

WARMUP_FREEZE  = True
WARMUP_EPOCHS  = 3

# ── Phase AB 超参（合并原 A+B，直接监控 drug4_acc）────────────────────────────
PHASE_AB_MAX_EP   = 25    # ★ v9: 原A(15)+B(20)合并，给25轮
PHASE_AB_THRESH   = 0.95  # ★ v9: 四分类比二分类难，阈值降至0.75
PHASE_CONSEC_REQ  = 2     # 连续达标轮数后切换 Phase C

DROPOUT       = 0.2
DROPPATH_RATE = 0.1
# ★ v9: nodrug 样本较少，加权处理类别不平衡 [cangrelor, aspirin, eptifibatide, nodrug]
DRUG4_WEIGHT  = [1.0, 1.0, 1.0, 1.0]
UNSHARP_P     = 0.3

W_DRUG4  = 1.0    # ★ v9: 四分类 CE 权重
W_TIME   = 0.4    # 时间辅助头 CE 权重（与 v8 一致）
W_SUPCON = 0.05    # Phase AB 不用 SupCon（Phase C 由 PhaseCLoss 控制）

SUPCON_TEMP       = 0.2
SUPCON_TIME_DELTA = 2

LR_PATIENCE = 3
LR_FACTOR   = 0.5
LR_MIN      = 1e-7

KNN_K_CONSISTENCY = 10
KNN_K_GRAPH       = 15
DIFFUSION_T       = 5
EMBED_CHUNK       = 512

for d in ['/content/data', PIPE_DIR, EMB_DIR, ANA_DIR, MT_DIR, TRAJ_DIR, FINAL_DIR]:
    os.makedirs(d, exist_ok=True)

print('=' * 72)
print(f'  显微血小板 Pipeline v9  —  {BACKBONE.upper()}')
print(f'  CROP={CROP_SIZE}  BATCH={BATCH_SIZE}  EMB_DIM={EMBEDDING_DIM}')
print(f'  ★ v9: 直接四分类，无 Phase A/B 层级结构')
print(f'  PhaseAB(drug4≥{PHASE_AB_THRESH}×{PHASE_CONSEC_REQ} or ep>{PHASE_AB_MAX_EP}) → PhaseC(+SupCon)')
print(f'  DropPath={DROPPATH_RATE}  Dropout={DROPOUT}  UnsharpP={UNSHARP_P}')
print(f'  Drug4Weight={DRUG4_WEIGHT}')
print(f'  Loss(PhaseAB): {W_DRUG4}×CE_drug4(weighted) + {W_TIME}×CE_time')
print('=' * 72)

# ── Phase C 超参（v12 设计完整保留）──────────────────────────────────────────
LR_C_BACKBONE   = 1e-6
LR_C_TIME_HEAD  = 5e-5
LR_C_SUPCON     = 1e-4

# WeightedTemporalSupConLoss (L4)
TC_SUPCON_T = 0.1
TC_TAU_S    = 1.5

# DrugStratifiedTripletLoss (L3)
TRIPLET_MARGIN = 0.5

# SmoothnessPenaltyLoss (L6)
SMOOTH_K           = 1.5
SMOOTH_DELTA_INIT  = 0.3
SMOOTH_UPDATE_FREQ = 5

# Phase C 消融开关
PHASE_C_USE_L3 = True
PHASE_C_USE_L6 = True

# Phase C Loss 权重
WC_L1_ORDINAL = 0.6
WC_L2_EMD     = 0.4
WC_L3_TRIPLET = 0.4
WC_L4_SUPCON  = 0.2
WC_L5_REPULSE = 0.2
WC_L6_SMOOTH  = 0.1

# Phase C Early Stopping
PATIENCE_C         = 10
DRUG4_ACC_GUARDIAN = 0.75   # ★ v9: 与 PHASE_AB_THRESH 对齐

print(f'  Phase C LR: bb={LR_C_BACKBONE:.0e}  th={LR_C_TIME_HEAD:.0e}  sc={LR_C_SUPCON:.0e}')
print(f'  Phase C Loss: L1*{WC_L1_ORDINAL}+L2*{WC_L2_EMD}+L4*{WC_L4_SUPCON}+L5*{WC_L5_REPULSE}', end='')
if PHASE_C_USE_L3: print(f'+L3*{WC_L3_TRIPLET}', end='')
if PHASE_C_USE_L6: print(f'+L6*{WC_L6_SMOOTH}', end='')
print()
print(f'  Phase C Guardian: drug4_acc_guardian={DRUG4_ACC_GUARDIAN}  patience_c={PATIENCE_C}')

  显微血小板 Pipeline v9  —  RESNET18
  CROP=512  BATCH=96  EMB_DIM=256
  ★ v9: 直接四分类，无 Phase A/B 层级结构
  PhaseAB(drug4≥0.95×2 or ep>25) → PhaseC(+SupCon)
  DropPath=0.1  Dropout=0.2  UnsharpP=0.3
  Drug4Weight=[1.0, 1.0, 1.0, 1.0]
  Loss(PhaseAB): 1.0×CE_drug4(weighted) + 0.4×CE_time
  Phase C LR: bb=1e-06  th=5e-05  sc=1e-04
  Phase C Loss: L1*0.6+L2*0.4+L4*0.2+L5*0.2+L3*0.4+L6*0.1
  Phase C Guardian: drug4_acc_guardian=0.75  patience_c=10


## Step 2 — HDF5 预打包（与 v8 相同）

In [ ]:
print('[Step 2] HDF5 文件已就绪，跳过打包')

[Step 2] HDF5 文件已就绪，跳过打包


## Step 3 — 数据集 & DataLoader（与 v8 相同）

In [ ]:
import random, numpy as np, h5py, torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision import transforms
from PIL import Image as PILImage, ImageFilter

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(SEED)


class PlateletHDF5Dataset(Dataset):
    def __init__(self, h5_path, crop_size=512, is_train=True, use_erasing=False):
        self.h5_path     = h5_path
        self.crop_size   = crop_size
        self.is_train    = is_train
        self.use_erasing = use_erasing
        self._handle     = None
        with h5py.File(h5_path, 'r') as f:
            self.n           = f['drug_label'].shape[0]
            self.drug_labels = f['drug_label'][:]
            self.time_labels = f['time_label'][:]
            ph_mean = float(f.attrs.get('phase_mean',  49.31))
            ph_std  = float(f.attrs.get('phase_std',   69.82))
            bf_mean = float(f.attrs.get('bf_mean',    147.67))
            bf_std  = float(f.attrs.get('bf_std',      18.92))
        self.ph_norm = ([ph_mean/255.0]*3,      [max(ph_std/255.0, 1e-4)]*3)
        self.bf_norm = ([bf_mean/255.0],        [max(bf_std/255.0,  1e-4)])

    def __len__(self): return self.n

    def _get_handle(self):
        if self._handle is None:
            self._handle = h5py.File(self.h5_path, 'r')
        return self._handle

    def _sync_spatial(self, ph_pil, bf_pil):
        i, j, h, w = transforms.RandomCrop.get_params(
            ph_pil, output_size=(self.crop_size, self.crop_size))
        ph_pil = TF.crop(ph_pil, i, j, h, w)
        bf_pil = TF.crop(bf_pil, i, j, h, w)
        if random.random() > 0.5: ph_pil, bf_pil = TF.hflip(ph_pil), TF.hflip(bf_pil)
        if random.random() > 0.5: ph_pil, bf_pil = TF.vflip(ph_pil), TF.vflip(bf_pil)
        angle = random.choice([0, 90, 180, 270])
        if angle != 0:
            ph_pil = TF.rotate(ph_pil, angle)
            bf_pil = TF.rotate(bf_pil, angle)
        return ph_pil, bf_pil

    def _phase_photometric(self, ph_pil):
        ph_pil = transforms.ColorJitter(
            brightness=0.2, contrast=0.3, saturation=0.10, hue=0.03)(ph_pil)
        if random.random() < 0.25:
            ph_pil = transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 0.8))(ph_pil)
        if random.random() < UNSHARP_P:
            ph_pil = ph_pil.filter(ImageFilter.UnsharpMask(radius=1, percent=80, threshold=2))
        return ph_pil

    def _phase_tensor_aug(self, ph_t, ph_raw_np):
        if random.random() < 0.3:
            ph_t = (ph_t + torch.randn_like(ph_t) * 0.03).clamp(-3, 3)
        if self.use_erasing:
            fg_ratio = float((ph_raw_np.mean(axis=2) > 15).mean())
            if fg_ratio > 0.03:
                eraser = transforms.RandomErasing(
                    p=1.0, scale=(0.005, 0.04), ratio=(0.5, 2.0), value=0)
                for _ in range(5):
                    cand = eraser(ph_t.clone())
                    if (cand != ph_t).float().mean() < 0.40:
                        ph_t = cand; break
        return ph_t

    def _augment(self, ph_pil, bf_pil):
        if self.is_train:
            ph_pil, bf_pil = self._sync_spatial(ph_pil, bf_pil)
            ph_pil = self._phase_photometric(ph_pil)
        else:
            ph_pil = TF.center_crop(ph_pil, (self.crop_size, self.crop_size))
            bf_pil = TF.center_crop(bf_pil, (self.crop_size, self.crop_size))
        return ph_pil, bf_pil

    def __getitem__(self, idx):
        try:
            f      = self._get_handle()
            ph_np  = f['phase'][idx]
            bf_np  = f['brightfield'][idx]
            ph_pil = PILImage.fromarray(ph_np.transpose(1, 2, 0))
            bf_pil = PILImage.fromarray(bf_np[0])
            ph_pil, bf_pil = self._augment(ph_pil, bf_pil)
            ph_t = TF.normalize(TF.to_tensor(ph_pil), self.ph_norm[0], self.ph_norm[1])
            bf_t = TF.normalize(TF.to_tensor(bf_pil), self.bf_norm[0], self.bf_norm[1])
            if self.is_train:
                ph_t = self._phase_tensor_aug(ph_t, np.array(ph_pil))
            return {'phase': ph_t, 'brightfield': bf_t,
                    'drug_label': torch.tensor(int(self.drug_labels[idx]), dtype=torch.long),
                    'time_label': torch.tensor(int(self.time_labels[idx]), dtype=torch.long)}
        except Exception:
            return self.__getitem__(random.randint(0, self.n - 1))


_kw = dict(num_workers=NUM_WORKERS, pin_memory=True,
           persistent_workers=(NUM_WORKERS > 0))
train_ds = PlateletHDF5Dataset(HDF5_TRAIN, CROP_SIZE, is_train=True)
val_ds   = PlateletHDF5Dataset(HDF5_VAL,   CROP_SIZE, is_train=False)
test_ds  = PlateletHDF5Dataset(HDF5_TEST,  CROP_SIZE, is_train=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, **_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **_kw)

print(f'  train : {len(train_ds):>6} samples  ({len(train_loader)} batches)')
print(f'  val   : {len(val_ds):>6} samples  ({len(val_loader)} batches)')
print(f'  test  : {len(test_ds):>6} samples  ({len(test_loader)} batches)')
drug_cnt = np.bincount(train_ds.drug_labels, minlength=N_DRUG)
time_cnt = np.bincount(train_ds.time_labels, minlength=N_TIME)
print('Drug 分布:', dict(zip(DRUG_CLASSES, drug_cnt.tolist())))
print('Time 分布:', dict(zip(TIME_CLASSES,  time_cnt.tolist())))


  train :  27023 samples  (281 batches)
  val   :   9074 samples  (95 batches)
  test  :   9031 samples  (95 batches)
Drug 分布: {'cangrelor': 6843, 'aspirin': 6032, 'eptifibatide': 6716, 'nodrug': 7432}
Time 分布: {'0min': 5401, '5min': 5445, '10min': 5611, '15min': 5456, '20min': 5110}


## Step 4 — 模型定义 v9
### ★ v9 改动
- 删除 `binary_head`（2分类）和 `drug3_head`（3分类）
- 新增 `drug4_head`（4分类，直接预测 cangrelor/aspirin/eptifibatide/nodrug）
- `forward` 返回 **4 个值**：`drug4_out, time_out, emb_cls, emb_con`
- 删除 `freeze_for_phase_a`, `unfreeze_for_phase_b`（不再有层级阶段）
- 保留 `freeze_for_phase_c`（Phase C：drug4_head 冻结，supcon+time_head 可训练）


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm, itertools
from torch.utils.checkpoint import checkpoint

BACKBONE_FEAT_DIM = {
    'resnet18': 512, 'resnet34': 512,
    'resnet50': 2048, 'resnet101': 2048, 'resnet152': 2048,
}
FEAT_DIM = BACKBONE_FEAT_DIM.get(BACKBONE, 512)
print(f'Backbone={BACKBONE}, FEAT_DIM={FEAT_DIM}')


class DropPath(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__(); self.drop_prob = drop_prob
    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training: return x
        keep = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        r = torch.rand(shape, dtype=x.dtype, device=x.device)
        return x * torch.floor(r + keep) / keep
    def extra_repr(self): return f'drop_prob={self.drop_prob:.3f}'


class BasicBlockWithDP(nn.Module):
    def __init__(self, block, drop_rate=0.0):
        super().__init__()
        self.conv1 = block.conv1; self.bn1 = block.bn1; self.relu = block.relu
        self.conv2 = block.conv2; self.bn2 = block.bn2
        self.downsample = block.downsample; self.stride = block.stride
        self.drop_path = DropPath(drop_rate) if drop_rate > 0.0 else nn.Identity()
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop_path(self.bn2(self.conv2(out)))
        if self.downsample is not None: identity = self.downsample(x)
        return self.relu(out + identity)


def _apply_droppath_to_resnet(base, drop_rate=0.1):
    layers = [base.layer1, base.layer2, base.layer3, base.layer4]
    all_blocks = [(l, n) for l in layers for n in list(l._modules.keys())]
    n_total = len(all_blocks)
    for idx, (layer, name) in enumerate(all_blocks):
        rate = drop_rate * (idx + 1) / n_total
        layer._modules[name] = BasicBlockWithDP(layer._modules[name], drop_rate=rate)
        print(f'  block [{idx}] {name} → DropPath(rate={rate:.4f})')
    return base


class ChannelExpand(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 3, 1, bias=False)
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_out', nonlinearity='relu')
    def forward(self, x): return self.conv(x)


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class FusionBlock(nn.Module):
    def __init__(self, feat_dim=512):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(feat_dim*2, feat_dim, 1, bias=False),
            nn.BatchNorm2d(feat_dim), nn.ReLU(inplace=True))
        self.se = SEBlock(feat_dim, 16)
        nn.init.kaiming_normal_(self.conv[0].weight, mode='fan_out', nonlinearity='relu')
    def forward(self, fp, fi): return self.se(self.conv(torch.cat([fp, fi], 1)))


class PlateletPipelineModel(nn.Module):
    """
    v9 双分支 ResNet18。
    ★ v9: 删除 binary_head + drug3_head，新增 drug4_head（直接4分类）。
           forward 返回 4 个值: (drug4_out, time_out, emb_cls, emb_con)。
    继承 v8: DropPath, Layer-SE, time_head, proj_supcon, Dropout(0.2)。
    """
    def __init__(self, backbone='resnet18', n_drug=4, n_time=5,
                 embedding_dim=256, pretrained=True,
                 dropout=0.2, drop_path_rate=0.1, use_checkpoint=True):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.embedding_dim  = embedding_dim
        self.feat_dim       = BACKBONE_FEAT_DIM.get(backbone, 512)
        fd = self.feat_dim

        weights_map = {
            'resnet18':  tvm.ResNet18_Weights.IMAGENET1K_V1,
            'resnet34':  tvm.ResNet34_Weights.IMAGENET1K_V1,
            'resnet50':  tvm.ResNet50_Weights.IMAGENET1K_V2,
            'resnet101': tvm.ResNet101_Weights.IMAGENET1K_V2,
        }
        base = getattr(tvm, backbone)(
            weights=weights_map.get(backbone) if pretrained else None)

        if drop_path_rate > 0.0 and backbone in ('resnet18', 'resnet34'):
            print(f'[v9] 应用 DropPath (rate={drop_path_rate}, 线性调度) ...')
            base = _apply_droppath_to_resnet(base, drop_path_rate)

        self.stem   = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        lc = {'resnet18': [64,128,256,512], 'resnet34': [64,128,256,512],
              'resnet50': [256,512,1024,2048], 'resnet101': [256,512,1024,2048]
              }.get(backbone, [64,128,256,512])

        self.layer1, self.se_l1 = base.layer1, SEBlock(lc[0])
        self.layer2, self.se_l2 = base.layer2, SEBlock(lc[1])
        self.layer3, self.se_l3 = base.layer3, SEBlock(lc[2])
        self.layer4, self.se_l4 = base.layer4, SEBlock(lc[3])

        self.bf_expand    = ChannelExpand()
        self.fusion       = FusionBlock(feat_dim=fd)
        self.gap          = nn.AdaptiveAvgPool2d(1)
        self.feat_dropout = nn.Dropout(dropout)

        def _proj():
            return nn.Sequential(
                nn.Linear(fd, 256), nn.BatchNorm1d(256),
                nn.ReLU(inplace=True), nn.Dropout(dropout),
                nn.Linear(256, embedding_dim))

        self.proj_cls    = _proj()
        self.proj_supcon = _proj()

        # ★ v9: drug4_head 直接4分类（替代原 binary_head + drug3_head）
        self.drug4_head = nn.Sequential(
            nn.Linear(embedding_dim, 128), nn.BatchNorm1d(128),
            nn.ReLU(inplace=True), nn.Dropout(dropout / 2),
            nn.Linear(128, n_drug))

        # 时间辅助分类头（保持 v8 设计）
        self.time_head = nn.Sequential(
            nn.Linear(embedding_dim, 64), nn.BatchNorm1d(64),
            nn.ReLU(inplace=True), nn.Dropout(dropout / 2),
            nn.Linear(64, n_time))

    def _encode_branch(self, x):
        def _run(t):
            t = self.stem(t)
            t = self.se_l1(self.layer1(t)); t = self.se_l2(self.layer2(t))
            t = self.se_l3(self.layer3(t)); t = self.se_l4(self.layer4(t))
            return t
        if self.use_checkpoint and self.training:
            return checkpoint(_run, x, use_reentrant=False)
        return _run(x)

    def _get_feat(self, phase, brightfield):
        fp   = self._encode_branch(phase)
        fi   = self._encode_branch(self.bf_expand(brightfield))
        feat = self.gap(self.fusion(fp, fi)).flatten(1)
        return self.feat_dropout(feat)

    def get_embedding(self, phase, brightfield):
        return F.normalize(self.proj_cls(self._get_feat(phase, brightfield)), dim=1)

    def forward(self, phase, brightfield):
        """返回 4 个值: (drug4_out, time_out, emb_cls, emb_con)"""
        feat    = self._get_feat(phase, brightfield)
        emb_cls = F.normalize(self.proj_cls(feat),    dim=1)
        emb_con = F.normalize(self.proj_supcon(feat), dim=1)
        return (self.drug4_head(emb_cls),
                self.time_head(emb_cls),
                emb_cls, emb_con)

    # ── 冻结控制 ────────────────────────────────────────────────────────────
    def freeze_backbone(self):
        for p in itertools.chain(self.stem.parameters(),
            self.layer1.parameters(), self.layer2.parameters(),
            self.layer3.parameters(), self.layer4.parameters()):
            p.requires_grad = False
        print('[Model] Backbone frozen')

    def unfreeze_backbone(self):
        for p in itertools.chain(self.stem.parameters(),
            self.layer1.parameters(), self.layer2.parameters(),
            self.layer3.parameters(), self.layer4.parameters()):
            p.requires_grad = True
        print('[Model] Backbone unfrozen')

    def freeze_for_phase_ab(self):
        """Phase AB: proj_supcon 冻结，drug4_head + time_head 可训练"""
        for p in self.proj_supcon.parameters(): p.requires_grad = True
        print('[Model] Phase AB: proj_supcon frozen')

    def freeze_for_phase_c(self):
        """Phase C (v12): drug4_head 完全冻结，supcon + time_head 可训练"""
        for p in self.drug4_head.parameters():  p.requires_grad = False
        #for p in self.proj_supcon.parameters(): p.requires_grad = False
        for p in self.time_head.parameters():   p.requires_grad = True
        print('[Model] Phase C: drug4_head FROZEN | supcon+time_head active')

    def unfreeze_for_phase_c(self):
        for p in self.proj_supcon.parameters(): p.requires_grad = True
        print('[Model] Phase C: proj_supcon unfrozen')

    def _bb_params(self):
        """返回 backbone (stem+layer1-4) 的参数列表，供 Phase C optimizer 使用。"""
        return list(itertools.chain(
            self.stem.parameters(),
            self.layer1.parameters(), self.layer2.parameters(),
            self.layer3.parameters(), self.layer4.parameters()))

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad = True
        print('[Model] All parameters unfrozen')

    def count_params(self):
        _n = lambda it: sum(p.numel() for p in it)
        return {
            'stem+layers (backbone)': _n(itertools.chain(
                self.stem.parameters(), self.layer1.parameters(),
                self.layer2.parameters(), self.layer3.parameters(),
                self.layer4.parameters())),
            'layer_SE (se_l1-4)':     _n(itertools.chain(
                self.se_l1.parameters(), self.se_l2.parameters(),
                self.se_l3.parameters(), self.se_l4.parameters())),
            'bf_expand+fusion':       _n(itertools.chain(
                self.bf_expand.parameters(), self.fusion.parameters())),
            'proj_cls':               _n(self.proj_cls.parameters()),
            'proj_supcon':            _n(self.proj_supcon.parameters()),
            'drug4_head':             _n(self.drug4_head.parameters()),
            'time_head':              _n(self.time_head.parameters()),
            'total':                  _n(self.parameters()),
            'trainable':              sum(p.numel() for p in self.parameters()
                                         if p.requires_grad),
        }


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = PlateletPipelineModel(
    backbone=BACKBONE, n_drug=N_DRUG, n_time=N_TIME,
    embedding_dim=EMBEDDING_DIM, pretrained=PRETRAINED,
    dropout=DROPOUT, drop_path_rate=DROPPATH_RATE, use_checkpoint=True,
).to(device)

pc = model.count_params()
print(f'\n参数统计 ({BACKBONE} + v9):')
for k, v in pc.items(): print(f'  {k:<30}: {v:>12,}')

# 前向验证（4输出）
model.eval()
with torch.no_grad():
    _ph = torch.randn(2, 3, CROP_SIZE, CROP_SIZE, device=device)
    _bf = torch.randn(2, 1, CROP_SIZE, CROP_SIZE, device=device)
    _dr4, _t, _ecls, _econ = model(_ph, _bf)
    print(f'\n前向验证:')
    print(f'  drug4={list(_dr4.shape)}  time={list(_t.shape)}')
    print(f'  emb_cls={list(_ecls.shape)}  norm={_ecls.norm(dim=1).mean():.4f}  ✓')
del _ph, _bf, _dr4, _t, _ecls, _econ
torch.cuda.empty_cache()


Backbone=resnet18, FEAT_DIM=512
Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]


[v9] 应用 DropPath (rate=0.1, 线性调度) ...
  block [0] 0 → DropPath(rate=0.0125)
  block [1] 1 → DropPath(rate=0.0250)
  block [2] 0 → DropPath(rate=0.0375)
  block [3] 1 → DropPath(rate=0.0500)
  block [4] 0 → DropPath(rate=0.0625)
  block [5] 1 → DropPath(rate=0.0750)
  block [6] 0 → DropPath(rate=0.0875)
  block [7] 1 → DropPath(rate=0.1000)

参数统计 (resnet18 + v9):
  stem+layers (backbone)        :   11,176,512
  layer_SE (se_l1-4)            :       43,520
  bf_expand+fusion              :      558,083
  proj_cls                      :      197,632
  proj_supcon                   :      197,632
  drug4_head                    :       33,668
  time_head                     :       16,901
  total                         :   12,223,948
  trainable                     :   12,223,948

前向验证:
  drug4=[2, 4]  time=[2, 5]
  emb_cls=[2, 256]  norm=1.0000  ✓


## Step 5 — 损失函数 v9
### ★ v9 改动
- 删除 `HierarchicalLoss`（依赖 binary/drug3 层级结构）
- 新增 `DirectDrug4Loss`：`W_DRUG4×CE_drug4(weighted) + W_TIME×CE_time`
- 删除 `hierarchical_accuracy`，新增 `direct_accuracy`（直接计算 drug4_acc）


In [ ]:
import torch, torch.nn as nn
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import f1_score, confusion_matrix as sk_cm, roc_auc_score, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class DirectDrug4Loss(nn.Module):
    """
    v9 直接四分类损失：
      Phase AB: W_DRUG4×CE_drug4(weighted) + W_TIME×CE_time
    """
    def __init__(self, w_drug4=1.0, w_time=0.4,
                 drug4_weight=None, device='cpu'):
        super().__init__()
        if drug4_weight is not None:
            dw = torch.tensor(drug4_weight, dtype=torch.float32, device=device)
            self.ce_drug4 = nn.CrossEntropyLoss(weight=dw)
        else:
            self.ce_drug4 = nn.CrossEntropyLoss()
        self.ce_time  = nn.CrossEntropyLoss()
        self.w_drug4  = w_drug4
        self.w_time   = w_time

    def forward(self, drug4_out, time_out, drug4_label, time_label):
        l_drug4 = self.ce_drug4(drug4_out, drug4_label)
        l_time  = self.ce_time(time_out,  time_label)
        total   = self.w_drug4 * l_drug4 + self.w_time * l_time
        return {'total': total, 'drug4': l_drug4, 'time': l_time}


class AverageMeter:
    def __init__(self): self.reset()
    def reset(self):    self.s = self.n = 0.0
    def update(self, v, n=1): self.s += float(v)*n; self.n += n
    @property
    def avg(self): return self.s / max(self.n, 1e-8)


def accuracy(logit, label):
    return (logit.argmax(1) == label).float().mean().item()


def direct_accuracy(drug4_out, time_out, drug4_label, time_label):
    """v9: 直接计算 drug4_acc 和 time_acc（无层级拆解）"""
    drug4_acc = accuracy(drug4_out, drug4_label)
    time_acc  = accuracy(time_out,  time_label)
    return drug4_acc, time_acc


def compute_full_metrics(pred, true, n_classes, class_names):
    pred, true = np.array(pred), np.array(true)
    acc  = accuracy_score(true, pred)
    f1   = f1_score(true, pred, average='macro', zero_division=0)
    cm   = sk_cm(true, pred, labels=list(range(n_classes)))
    try:
        from sklearn.preprocessing import label_binarize
        tb = label_binarize(true, classes=list(range(n_classes)))
        pb = label_binarize(pred, classes=list(range(n_classes)))
        auroc = roc_auc_score(tb, pb, multi_class='ovr', average='macro')
    except Exception: auroc = float('nan')
    return {'acc': acc, 'f1': f1, 'auroc': auroc, 'cm': cm}


def plot_confusion_matrix(cm, class_names, title='', ax=None, save_path=None):
    if ax is None: fig, ax = plt.subplots(figsize=(7, 6))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    if save_path: plt.savefig(save_path, bbox_inches='tight', dpi=150)
    return ax


criterion = DirectDrug4Loss(
    w_drug4=W_DRUG4, w_time=W_TIME,
    drug4_weight=DRUG4_WEIGHT, device=str(device),
).to(device)

print('损失函数: DirectDrug4Loss v9')
print(f'  Phase AB: {W_DRUG4}×CE_drug4(weight={DRUG4_WEIGHT}) + {W_TIME}×CE_time')
print(f'  Phase C:  PhaseCLoss (L1~L6, 见 Step 5b)')


损失函数: DirectDrug4Loss v9
  Phase AB: 1.0×CE_drug4(weight=[1.0, 1.0, 1.0]) + 0.4×CE_time
  Phase C:  PhaseCLoss (L1~L6, 见 Step 5b)


## Step 5b — Phase C 损失函数（v12 完整保留）

★ Phase C 使用 `PhaseCLoss`（L1~L6 组合），与 v8/v12 完全一致。
`freeze_for_phase_c` 已在 Step 4 模型中定义（drug4_head FROZEN）。


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from scipy.stats import kendalltau
from sklearn.metrics.pairwise import cosine_distances

# ── L1: OrdinalCELoss ────────────────────────────────────────────────────────
class OrdinalCELoss(nn.Module):
    """L1: CE + squared-distance penalty."""
    def __init__(self, n=5, alpha=0.6):
        super().__init__()
        self.alpha = alpha
        D = torch.zeros(n, n)
        for i in range(n):
            for j in range(n):
                D[i, j] = (i - j) ** 2
        self.register_buffer('D', D)

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets)
        probs = F.softmax(logits, dim=1)
        return ce + self.alpha * (probs * self.D[targets]).sum(1).mean()


# ── L2: EarthMoverLoss ───────────────────────────────────────────────────────
class EarthMoverLoss(nn.Module):
    """L2: CDF distance (1D EMD)."""
    def forward(self, logits, targets):
        B, C  = logits.shape
        probs = F.softmax(logits, dim=1)
        oh    = F.one_hot(targets, C).float()
        return (torch.cumsum(probs, dim=1) - torch.cumsum(oh, dim=1)).abs().mean()


# ── L3: DrugStratifiedTripletLoss  [可消融: PHASE_C_USE_L3=False] ────────────
class DrugStratifiedTripletLoss(nn.Module):
    """L3: Drug-stratified triplet."""
    def __init__(self, margin=0.5):
        super().__init__()
        self.m = margin

    def forward(self, emb, drug, time):
        zero   = torch.tensor(0.0, device=emb.device)
        losses = []
        for d in torch.unique(drug):
            mk = (drug == d)
            if mk.sum() < 3:
                continue
            ed   = emb[mk]
            td   = time[mk]
            n    = ed.size(0)
            dist = torch.cdist(ed.unsqueeze(0), ed.unsqueeze(0)).squeeze(0)
            ta   = td.unsqueeze(1).expand(n, n)
            tb   = td.unsqueeze(0).expand(n, n)
            dt   = (ta - tb).abs()
            for i in range(n):
                pi = torch.where(dt[i] == 1)[0]
                ni = torch.where(dt[i] == 2)[0]
                if len(pi) == 0 or len(ni) == 0:
                    continue
                losses.append(F.relu(dist[i][pi].mean() - dist[i][ni].mean() + self.m))
        return torch.stack(losses).mean() if losses else zero


# ── L4: WeightedTemporalSupConLoss ───────────────────────────────────────────
class WeightedTemporalSupConLoss(nn.Module):
    """L4: Weighted TemporalSupCon — Phase C core loss."""
    def __init__(self, T=0.1, tau_s=1.5):
        super().__init__()
        self.T     = T
        self.tau_s = tau_s

    def forward(self, emb, drug, time):
        B = emb.size(0)
        if B < 4:
            return emb.sum() * 0.0
        sim   = torch.matmul(emb, emb.T) / self.T
        deq   = (drug.unsqueeze(0) == drug.unsqueeze(1))
        ta    = time.unsqueeze(0).expand(B, B).float()
        tb    = time.unsqueeze(1).expand(B, B).float()
        dt    = (ta - tb).abs()
        w     = torch.exp(-dt / self.tau_s)
        pos   = deq & (dt <= 2);  pos.fill_diagonal_(False)
        neg   = (~deq) | (deq & (dt >= 3)); neg.fill_diagonal_(False)
        valid = pos.any(dim=1) & neg.any(dim=1)
        if not valid.any():
            return emb.sum() * 0.0
        exp_sim = torch.exp(sim.clamp(max=30.0))
        neg_sum = (exp_sim * neg.float()).sum(dim=1).clamp(min=1e-6)
        lpp     = sim - torch.log(neg_sum.unsqueeze(1))
        wpos    = lpp * pos.float() * w
        wp_sum  = (pos.float() * w).sum(dim=1).clamp(min=1e-6)
        return (-wpos.sum(dim=1) / wp_sum)[valid].mean()


# ── L5: CrossDrugRepulsionLoss ───────────────────────────────────────────────
class CrossDrugRepulsionLoss(nn.Module):
    """L5: Push different-drug embeddings apart."""
    def __init__(self, margin=0.5):
        super().__init__()
        self.m = margin

    def forward(self, emb, drug):
        zero = torch.tensor(0.0, device=emb.device)
        neq  = (drug.unsqueeze(0) != drug.unsqueeze(1))
        neq.fill_diagonal_(False)
        if not neq.any():
            return zero
        cos = torch.matmul(emb, emb.T)
        pen = F.relu(cos - (1.0 - self.m))
        return (pen * neq.float()).sum() / neq.float().sum().clamp(min=1)


# ── L6: SmoothnessPenaltyLoss  [可消融: PHASE_C_USE_L6=False] ────────────────
class SmoothnessPenaltyLoss(nn.Module):
    """L6: Centroid smoothness upper bound."""
    def __init__(self, n=5, k=1.5, d_init=0.3):
        super().__init__()
        self.n         = n
        self.k         = k
        self.delta_max = d_init

    def update_delta(self, emb, drug, time):
        with torch.no_grad():
            diffs = []
            for d in torch.unique(drug):
                mk = (drug == d)
                ed = emb[mk]; td = time[mk]
                cs = [ed[td == t].mean(0) for t in range(self.n)
                      if (td == t).sum() > 0]
                diffs += [torch.norm(cs[i + 1] - cs[i]).item()
                          for i in range(len(cs) - 1)]
            if diffs:
                self.delta_max = self.k * float(np.mean(diffs))
        return self.delta_max

    def forward(self, emb, drug, time):
        zero   = torch.tensor(0.0, device=emb.device)
        losses = []
        for d in torch.unique(drug):
            mk = (drug == d)
            ed = emb[mk]; td = time[mk]
            cs = [ed[td == t].mean(0) for t in range(self.n)
                  if (td == t).sum() > 0]
            losses += [F.relu(torch.norm(cs[i + 1] - cs[i]) - self.delta_max)
                       for i in range(len(cs) - 1)]
        return torch.stack(losses).mean() if losses else zero


# ── PhaseCLoss: L1 + L2 + L4 + L5 [+ L3] [+ L6] ────────────────────────────
class PhaseCLoss(nn.Module):
    """v12 Phase C total loss: L1+L2+L4+L5 [+L3] [+L6]"""
    def __init__(self, n=5,
                 w1=0.6, w2=0.4, w3=0.4, w4=0.5, w5=0.2, w6=0.1,
                 tc_T=0.1, tau_s=1.5, tm=0.5,
                 use_l3=True, use_l6=True, k=1.5, d_init=0.3):
        super().__init__()
        self.w1=w1; self.w2=w2; self.w3=w3
        self.w4=w4; self.w5=w5; self.w6=w6
        self.use_l3 = use_l3
        self.use_l6 = use_l6
        self.l1 = OrdinalCELoss(n, alpha=0.6)
        self.l2 = EarthMoverLoss()
        self.l3 = DrugStratifiedTripletLoss(tm)
        self.l4 = WeightedTemporalSupConLoss(tc_T, tau_s)
        self.l5 = CrossDrugRepulsionLoss(margin=0.5)
        self.l6 = SmoothnessPenaltyLoss(n, k, d_init)

    def forward(self, time_out, emb_con, drug, time):
        """
        time_out : [B, N_TIME]  time_head logits
        emb_con  : [B, D]       proj_supcon output (L2-normalized)
        drug     : [B]          4-class drug label (0-3)
        time     : [B]          5-class time label (0-4)
        """
        zero  = torch.tensor(0.0, device=time_out.device)
        l1    = self.l1(time_out, time)
        l2    = self.l2(time_out, time)
        l3    = self.l3(emb_con, drug, time) if self.use_l3 else zero
        l4    = self.l4(emb_con, drug, time)
        l5    = self.l5(emb_con, drug)
        l6    = self.l6(emb_con, drug, time) if self.use_l6 else zero
        total = self.w1*l1 + self.w2*l2 + self.w4*l4 + self.w5*l5
        if self.use_l3: total = total + self.w3*l3
        if self.use_l6: total = total + self.w6*l6
        return {
            'total':      total,
            'l1_ordinal': l1,
            'l2_emd':     l2,
            'l3_triplet': l3,
            'l4_supcon':  l4,
            'l5_repulse': l5,
            'l6_smooth':  l6,
        }


# ── 验证指标：Kendall tau + Temporal Ordering Accuracy ───────────────────────
def compute_temporal_metrics_fast(emb_np, drug_np, time_np,
                                   n_drug=4, n_time=5, seed=42):
    rng = np.random.default_rng(seed)
    tau_list = []
    for d in range(n_drug):
        mk = (drug_np == d)
        if mk.sum() < 5:
            continue
        ed = emb_np[mk]; td = time_np[mk]; n = len(ed)
        dm = cosine_distances(ed)
        dists, dts = [], []
        for i in range(n):
            for j in range(i + 1, n):
                dists.append(dm[i, j])
                dts.append(abs(int(td[i]) - int(td[j])))
        if len(dists) > 5:
            tau, _ = kendalltau(dts, dists)
            if not np.isnan(tau):
                tau_list.append(tau)
    mean_tau = float(np.mean(tau_list)) if tau_list else 0.0

    ok = tot = 0
    for d in range(n_drug):
        idx = np.where(drug_np == d)[0]
        if len(idx) < 3:
            continue
        ed = emb_np[idx]; td = time_np[idx]
        dm = cosine_distances(ed); n = len(ed)
        si = rng.choice(n, min(n, 30), replace=False)
        for ii in si:
            for jj in range(n):
                for kk in range(n):
                    if ii == jj or ii == kk or jj == kk:
                        continue
                    dij = abs(int(td[ii]) - int(td[jj]))
                    dik = abs(int(td[ii]) - int(td[kk]))
                    if dij < dik:
                        tot += 1
                        ok  += (1 if dm[ii, jj] < dm[ii, kk] else 0)
    ord_acc = ok / max(tot, 1)
    return mean_tau, ord_acc


# ── 实例化验证 ────────────────────────────────────────────────────────────────
_test_loss_c = PhaseCLoss(
    n=N_TIME,
    w1=WC_L1_ORDINAL, w2=WC_L2_EMD,    w3=WC_L3_TRIPLET,
    w4=WC_L4_SUPCON,  w5=WC_L5_REPULSE, w6=WC_L6_SMOOTH,
    tc_T=TC_SUPCON_T,  tau_s=TC_TAU_S,   tm=TRIPLET_MARGIN,
    use_l3=PHASE_C_USE_L3, use_l6=PHASE_C_USE_L6,
    k=SMOOTH_K, d_init=SMOOTH_DELTA_INIT,
).to(device)

print('✅ Step 5b: PhaseCLoss 实例化成功')
print(f'   L1*{WC_L1_ORDINAL} + L2*{WC_L2_EMD} + L4*{WC_L4_SUPCON} + L5*{WC_L5_REPULSE}', end='')
if PHASE_C_USE_L3: print(f' + L3*{WC_L3_TRIPLET}', end='')
if PHASE_C_USE_L6: print(f' + L6*{WC_L6_SMOOTH}', end='')
print()
del _test_loss_c


✅ Step 5b: PhaseCLoss 实例化成功
   L1*0.6 + L2*0.4 + L4*0.2 + L5*0.2 + L3*0.4 + L6*0.1


## Step 6 — 两阶段训练 v9
### ★ v9 改动
- `forward` 拆包 **4 个值**：`(drug4_out, time_out, emb_cls, emb_con)`
- `_train_one_epoch` / `_validate`：使用 `DirectDrug4Loss`，监控 `drug4_acc` + `time_acc`
- `_rebuild_optimizer_for_phase_ab`：统一 `drug4_time_head` 参数组（无 binary/drug3）
- 单次相位切换：Phase AB → Phase C（`drug4_acc≥PHASE_AB_THRESH` × 2 或 `epoch>PHASE_AB_MAX_EP`）
- Phase C 训练循环、optimizer 重建、早停、Guardian 全部保留 v12 原设计


In [ ]:
import os, gc, time, itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from datetime import datetime
from tqdm import tqdm

os.makedirs(MT_DIR, exist_ok=True)
MT_CKPT      = os.path.join(MT_DIR, 'best_mt_v9.pth')
MT_CKPT_LAST = os.path.join(MT_DIR, 'latest_mt_v9.pth')
MT_LOG       = os.path.join(MT_DIR, 'train_log_v9.csv')

# ★ v9: 删除 binary/drug3 列，新增 drug4 列
_LOG_HEADER_AB = [
    'epoch', 'phase',
    'trn_loss', 'trn_drug4', 'trn_time',
    'trn_drug4_acc', 'trn_time_acc',
    'val_loss', 'val_drug4', 'val_time',
    'val_drug4_acc', 'val_time_acc',
    'lr_backbone', 'lr_head', 'elapsed_s', 'timestamp',
]
_LOG_HEADER_C = [
    'epoch', 'phase',
    'trn_loss', 'trn_l1', 'trn_l2', 'trn_l3', 'trn_l4', 'trn_l5', 'trn_l6',
    'trn_time_acc', 'trn_drug4_acc',
    'gnorm_supcon', 'gnorm_time_head', 'gnorm_backbone',
    'val_loss', 'val_drug4_acc', 'val_time_acc',
    'val_kendall_tau', 'val_temporal_ord', 'smooth_delta',
    'lr_backbone', 'lr_supcon', 'lr_time_head', 'elapsed_s', 'timestamp',
]


def _train_one_epoch_ab(model, loader, criterion, optimizer, scaler, device, epoch):
    """Phase AB 专用 train loop：DirectDrug4Loss。"""
    model.train()
    meters = {k: AverageMeter() for k in
              ['total', 'drug4', 'time', 'drug4_acc', 'time_acc']}
    pbar = tqdm(loader, desc=f'Train E{epoch:>3d} [AB]', leave=False, dynamic_ncols=True)
    for batch in pbar:
        ph   = batch['phase'].to(device, non_blocking=True)
        bf   = batch['brightfield'].to(device, non_blocking=True)
        d_lb = batch['drug_label'].to(device, non_blocking=True)
        t_lb = batch['time_label'].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16,
                            enabled=torch.cuda.is_available()):
            drug4_out, time_out, _, emb_con = model(ph, bf)
            losses = criterion(drug4_out, time_out, d_lb, t_lb)
        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer); scaler.update()
        bsz = ph.size(0)
        for k in ['total', 'drug4', 'time']:
            meters[k].update(losses[k].item(), bsz)
        d4_acc, t_acc = direct_accuracy(
            drug4_out.detach(), time_out.detach(), d_lb, t_lb)
        meters['drug4_acc'].update(d4_acc, bsz)
        meters['time_acc'].update(t_acc,  bsz)
        pbar.set_postfix(
            loss=f"{meters['total'].avg:.3f}",
            dr4=f"{meters['drug4_acc'].avg:.3f}",
            t=f"{meters['time_acc'].avg:.3f}")
    return {k: m.avg for k, m in meters.items()}


@torch.no_grad()
def _validate_ab(model, loader, criterion, device):
    """Phase AB 专用 validate。"""
    model.eval()
    meters = {k: AverageMeter() for k in
              ['total', 'drug4', 'time', 'drug4_acc', 'time_acc']}
    for batch in tqdm(loader, desc='Val [AB]', leave=False):
        ph   = batch['phase'].to(device, non_blocking=True)
        bf   = batch['brightfield'].to(device, non_blocking=True)
        d_lb = batch['drug_label'].to(device, non_blocking=True)
        t_lb = batch['time_label'].to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16,
                             enabled=torch.cuda.is_available()):
            drug4_out, time_out, _, emb_con = model(ph, bf)
            losses = criterion(drug4_out, time_out, d_lb, t_lb)
        bsz = ph.size(0)
        for k in ['total', 'drug4', 'time']:
            v = losses[k].item()
            if v == v: meters[k].update(v, bsz)
        d4_acc, t_acc = direct_accuracy(drug4_out, time_out, d_lb, t_lb)
        meters['drug4_acc'].update(d4_acc, bsz)
        meters['time_acc'].update(t_acc,  bsz)
    return {k: m.avg for k, m in meters.items()}


def _train_one_epoch_c(model, loader, criterion_c, optimizer, scaler, device, epoch):
    """Phase C 专用 train loop：只用 PhaseCLoss，不再计算 drug4 分类损失。"""
    model.train()
    keys = ['total', 'l1_ordinal', 'l2_emd', 'l3_triplet',
            'l4_supcon', 'l5_repulse', 'l6_smooth',
            'time_acc', 'drug4_acc',
            'gnorm_supcon', 'gnorm_time_head', 'gnorm_backbone']
    meters = {k: AverageMeter() for k in keys}
    pbar   = tqdm(loader, desc=f'Train E{epoch:>3d} [C]', leave=False, dynamic_ncols=True)

    for batch in pbar:
        ph   = batch['phase'].to(device, non_blocking=True)
        bf   = batch['brightfield'].to(device, non_blocking=True)
        d_lb = batch['drug_label'].to(device, non_blocking=True)
        t_lb = batch['time_label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16,
                            enabled=torch.cuda.is_available()):
            drug4_out, time_out, _, emb_con = model(ph, bf)
            losses = criterion_c(time_out, emb_con, d_lb, t_lb)

        scaler.scale(losses['total']).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        bsz = ph.size(0)
        for k in ['total', 'l1_ordinal', 'l2_emd', 'l3_triplet',
                  'l4_supcon', 'l5_repulse', 'l6_smooth']:
            meters[k].update(losses[k].item(), bsz)
        meters['time_acc'].update(
            (time_out.detach().argmax(1) == t_lb).float().mean().item(), bsz)
        d4_acc, _ = direct_accuracy(drug4_out.detach(), time_out.detach(), d_lb, t_lb)
        meters['drug4_acc'].update(d4_acc, bsz)

        def _gnorm(params):
            gs = [p.grad.norm().item() for p in params if p.grad is not None]
            return float(np.mean(gs)) if gs else 0.0
        meters['gnorm_supcon'].update(_gnorm(model.proj_supcon.parameters()), 1)
        meters['gnorm_time_head'].update(_gnorm(model.time_head.parameters()), 1)
        meters['gnorm_backbone'].update(_gnorm(model.layer4.parameters()), 1)

        pbar.set_postfix(
            tot=f"{meters['total'].avg:.3f}",
            l4=f"{meters['l4_supcon'].avg:.4f}",
            t=f"{meters['time_acc'].avg:.3f}",
            dr4=f"{meters['drug4_acc'].avg:.3f}",
            gnSC=f"{meters['gnorm_supcon'].avg:.3f}")

    result = {k: m.avg for k, m in meters.items()}
    if result['gnorm_supcon'] > 3.0:
        print(f'  ⚠ WARNING gnorm_supcon={result["gnorm_supcon"]:.3f}>3.0')
        print(f'    → 考虑降低 WC_L4_SUPCON（当前={WC_L4_SUPCON}）')
    return result


@torch.no_grad()
def _validate_c(model, loader, criterion_c, device, epoch, do_smooth=False):
    """Phase C 专用 validate：PhaseCLoss + Kendall tau + temporal ordering acc。"""
    model.eval()
    keys   = ['total', 'l1_ordinal', 'l2_emd', 'l3_triplet',
              'l4_supcon', 'l5_repulse', 'l6_smooth',
              'time_acc', 'drug4_acc']
    meters = {k: AverageMeter() for k in keys}
    all_emb, all_drug, all_time = [], [], []

    for batch in tqdm(loader, desc='Val [C]', leave=False):
        ph   = batch['phase'].to(device, non_blocking=True)
        bf   = batch['brightfield'].to(device, non_blocking=True)
        d_lb = batch['drug_label'].to(device, non_blocking=True)
        t_lb = batch['time_label'].to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16,
                            enabled=torch.cuda.is_available()):
            drug4_out, time_out, _, emb_con = model(ph, bf)
            losses = criterion_c(time_out, emb_con, d_lb, t_lb)

        bsz = ph.size(0)
        for k in ['total', 'l1_ordinal', 'l2_emd', 'l3_triplet',
                  'l4_supcon', 'l5_repulse', 'l6_smooth']:
            v = losses[k].item()
            if v == v:
                meters[k].update(v, bsz)
        meters['time_acc'].update(
            (time_out.argmax(1) == t_lb).float().mean().item(), bsz)
        d4_acc, _ = direct_accuracy(drug4_out, time_out, d_lb, t_lb)
        meters['drug4_acc'].update(d4_acc, bsz)

        all_emb.append(emb_con.float().cpu().numpy())
        all_drug.append(d_lb.cpu().numpy())
        all_time.append(t_lb.cpu().numpy())

    emb_np  = np.concatenate(all_emb,  axis=0)
    drug_np = np.concatenate(all_drug, axis=0)
    time_np = np.concatenate(all_time, axis=0)

    tau, ord_acc = compute_temporal_metrics_fast(
        emb_np, drug_np, time_np, n_drug=N_DRUG, n_time=N_TIME)

    result                       = {k: m.avg for k, m in meters.items()}
    result['kendall_tau']        = tau
    result['temporal_ord_acc']   = ord_acc
    result['smooth_delta']       = criterion_c.l6.delta_max

    print(f'  [Val C E{epoch}] tot={result["total"]:.4f}  l4={result["l4_supcon"]:.4f}  '
          f'dr4={result["drug4_acc"]:.4f}  time={result["time_acc"]:.4f}  '
          f'τ={tau:.4f}  ord={ord_acc:.4f}')

    if do_smooth and PHASE_C_USE_L6:
        et = torch.from_numpy(emb_np).to(device)
        dt = torch.from_numpy(drug_np).to(device)
        tt = torch.from_numpy(time_np).to(device)
        new_d = criterion_c.l6.update_delta(et, dt, tt)
        result['smooth_delta'] = new_d
        print(f'    [L6] delta_max updated → {new_d:.4f}')

    return result


def _rebuild_optimizer_for_phase_ab(model, warmup_done):
    """
    Phase AB optimizer 参数组。
    ★ v9: drug4_head + time_head 合并为一个参数组 'drug4_time_head'。

      warmup=F : [heads]                          → 1组
      warmup=T : [heads, backbone]                → 2组
      药4+time 始终以 LR_HEAD*0.5 的组存在       → +1组
    """
    # heads: bf_expand, fusion, SE blocks, proj_cls（基础特征提取头）
    head_params = []
    for m in [model.bf_expand, model.fusion,
              model.se_l1, model.se_l2, model.se_l3, model.se_l4,
              model.proj_cls]:
        head_params += list(m.parameters())
    groups = [{'params': head_params, 'lr': LR_HEAD,
               'name': 'heads', 'weight_decay': WEIGHT_DECAY}]
    # drug4_time_head: 分类头以较小 LR 训练，避免初期过拟合
    d4t = list(model.drug4_head.parameters()) + list(model.time_head.parameters())
    groups.append({'params': d4t, 'lr': LR_HEAD * 0.5,
                   'name': 'drug4_time_head', 'weight_decay': WEIGHT_DECAY})
    groups.append({'params': list(model.proj_supcon.parameters()),
               'lr': LR_HEAD * 0.5,
               'name': 'proj_supcon', 'weight_decay': WEIGHT_DECAY})
    if warmup_done:
        bb = list(model.stem.parameters())
        for l in [model.layer1, model.layer2, model.layer3, model.layer4]:
            bb += list(l.parameters())
        groups.append({'params': bb, 'lr': LR_BACKBONE,
                       'name': 'backbone', 'weight_decay': WEIGHT_DECAY})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def train_v9(resume=False):
    print('=' * 76)
    print(f'  v9 两阶段训练: {BACKBONE.upper()}')
    print(f'  Phase AB: drug4直接四分类 → drug4≥{PHASE_AB_THRESH}×{PHASE_CONSEC_REQ} or ep>{PHASE_AB_MAX_EP}')
    print(f'  Phase C:  +SupCon时间结构化 → EarlyStopping(patience={PATIENCE_C})')
    print(f'  BATCH={BATCH_SIZE}  LR_bb={LR_BACKBONE:.1e}  LR_hd={LR_HEAD:.1e}')
    print(f'  ★ Drug4Weight={DRUG4_WEIGHT}  W_TIME={W_TIME}')
    print('=' * 76)

    for p in [HDF5_TRAIN, HDF5_VAL]:
        if not os.path.exists(p): raise FileNotFoundError(f'HDF5 不存在: {p}')

    _model = PlateletPipelineModel(
        backbone=BACKBONE, n_drug=N_DRUG, n_time=N_TIME,
        embedding_dim=EMBEDDING_DIM, pretrained=PRETRAINED,
        dropout=DROPOUT, drop_path_rate=DROPPATH_RATE, use_checkpoint=True,
    ).to(device)

    _criterion = DirectDrug4Loss(
        w_drug4=W_DRUG4, w_time=W_TIME,
        drug4_weight=DRUG4_WEIGHT, device=str(device),
    ).to(device)

    _criterion_c = PhaseCLoss(
        n=N_TIME,
        w1=WC_L1_ORDINAL, w2=WC_L2_EMD,    w3=WC_L3_TRIPLET,
        w4=WC_L4_SUPCON,  w5=WC_L5_REPULSE, w6=WC_L6_SMOOTH,
        tc_T=TC_SUPCON_T,  tau_s=TC_TAU_S,   tm=TRIPLET_MARGIN,
        use_l3=PHASE_C_USE_L3, use_l6=PHASE_C_USE_L6,
        k=SMOOTH_K, d_init=SMOOTH_DELTA_INIT,
    ).to(device)

    # Phase AB 初始状态：冻结 proj_supcon，backbone warmup
    for p in _model.proj_supcon.parameters(): p.requires_grad = True  # AB 阶段直接启动
    if WARMUP_FREEZE: _model.freeze_backbone()

    start_epoch   = 1
    best_val_acc  = 0.0
    log_rows      = []
    warmup_done   = False
    current_phase = 'AB'
    phase_consec  = 0
    # Phase C 专用状态
    best_tau = -1.0
    pat_c    = 0
    guard_c  = 0

    if not resume or not os.path.exists(MT_LOG):
        pd.DataFrame(columns=_LOG_HEADER_AB).to_csv(MT_LOG, index=False)

    if resume and os.path.exists(MT_CKPT_LAST):
        print(f'\n[断点续训] 加载 {MT_CKPT_LAST} ...')
        ck = torch.load(MT_CKPT_LAST, map_location='cpu')
        start_epoch   = ck['epoch'] + 1
        best_val_acc  = ck.get('best_val_acc', 0.0)
        log_rows      = ck.get('log_rows', [])
        warmup_done   = ck.get('warmup_done', False)
        current_phase = ck.get('phase', 'AB')
        phase_consec  = ck.get('phase_consec', 0)
        _model.load_state_dict(ck['model'])
        # 恢复冻结状态
        if current_phase == 'AB':
            for p in _model.proj_supcon.parameters(): p.requires_grad = True  # AB 阶段直接启动
            if not warmup_done: _model.freeze_backbone()
        else:  # C
            _model.freeze_for_phase_c()
        # 重建匹配的 optimizer
        if current_phase == 'C':
            bb_all = list(_model._bb_params())
            for _m in [_model.se_l1, _model.se_l2, _model.se_l3, _model.se_l4,
                        _model.bf_expand, _model.fusion, _model.proj_cls]:
                bb_all += list(_m.parameters())
            optimizer = torch.optim.AdamW([
                {'params': bb_all,
                 'lr': LR_C_BACKBONE,  'name': 'backbone',  'weight_decay': WEIGHT_DECAY},
                {'params': list(_model.time_head.parameters()),
                 'lr': LR_C_TIME_HEAD, 'name': 'time_head', 'weight_decay': WEIGHT_DECAY},
                {'params': list(_model.proj_supcon.parameters()),
                 'lr': LR_C_SUPCON,   'name': 'supcon',    'weight_decay': WEIGHT_DECAY},
            ], weight_decay=WEIGHT_DECAY)
        else:
            optimizer = _rebuild_optimizer_for_phase_ab(_model, warmup_done)
        try:
            optimizer.load_state_dict(ck['optimizer'])
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(device)
            print(f'  ✅ 优化器加载成功 (phase={current_phase}, '
                  f'groups={len(optimizer.param_groups)}, warmup={warmup_done})')
        except Exception as e:
            print(f'  ⚠️ 优化器加载失败: {e}  → 使用初始状态')
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR,
            patience=LR_PATIENCE, min_lr=LR_MIN, threshold=1e-4)
        scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
        try:
            scheduler.load_state_dict(ck['scheduler'])
            scaler.load_state_dict(ck['scaler'])
        except Exception: pass
        if current_phase == 'C':
            best_tau = ck.get('best_tau', -1.0)
            pat_c    = ck.get('pat_c',   0)
            guard_c  = ck.get('guard_c', 0)
        del ck; gc.collect()
        print(f'  续训自 epoch={start_epoch}  phase={current_phase}  best={best_val_acc:.4f}')
    else:
        # 新训练：初始 optimizer（无 backbone，等待 warmup）
        optimizer = _rebuild_optimizer_for_phase_ab(_model, warmup_done=False)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR,
            patience=LR_PATIENCE, min_lr=LR_MIN, threshold=1e-4)
        scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    print(f'\n开始训练: epoch {start_epoch} → {EPOCHS_MT}\n')
    print(f'{"Ep":>4} {"Ph":>4} {"TrnLoss":>8} {"TrnDr4":>7} {"TrnTime":>8} '
          f'{"ValLoss":>8} {"ValDr4":>7} {"ValTime":>8} {"LR_bb":>8} {"LR_hd":>8} {"s":>5}')
    print('-' * 90)

    for epoch in range(start_epoch, EPOCHS_MT + 1):

        # ── Warmup 解冻 backbone ─────────────────────────────────────────────
        if WARMUP_FREEZE and not warmup_done and epoch > WARMUP_EPOCHS:
            _model.unfreeze_backbone(); warmup_done = True
            bb = list(_model.stem.parameters())
            for l in [_model.layer1, _model.layer2, _model.layer3, _model.layer4]:
                bb += list(l.parameters())
            optimizer.add_param_group({'params': bb, 'lr': LR_BACKBONE,
                                       'name': 'backbone', 'weight_decay': WEIGHT_DECAY})
            trainable = sum(p.numel() for p in _model.parameters() if p.requires_grad)
            print(f'\n[E{epoch}] Warmup → backbone 解冻  trainable={trainable:,}  '
                  f'groups={len(optimizer.param_groups)}\n')

        t0 = time.time()

        if current_phase == 'AB':
            trn = _train_one_epoch_ab(_model, train_loader, _criterion,
                                      optimizer, scaler, device, epoch)
            val = _validate_ab(_model, val_loader, _criterion, device)
            scheduler.step(val['total'])
        else:  # Phase C
            do_smooth = (epoch % SMOOTH_UPDATE_FREQ == 0)
            trn = _train_one_epoch_c(_model, train_loader, _criterion_c,
                                     optimizer, scaler, device, epoch)
            val = _validate_c(_model, val_loader, _criterion_c, device,
                               epoch, do_smooth=do_smooth)

        elapsed = time.time() - t0

        # ── Phase AB → C 切换 ────────────────────────────────────────────────
        if current_phase == 'AB':
            if val['drug4_acc'] >= PHASE_AB_THRESH:
                phase_consec += 1
            else:
                phase_consec = 0
            if phase_consec >= PHASE_CONSEC_REQ or epoch >= PHASE_AB_MAX_EP:
                current_phase = 'C'; phase_consec = 0; pat_c = 0; guard_c = 0
                # v12 Phase C 冻结策略
                _model.freeze_for_phase_c()
                # 完全重建 optimizer（3 组精细 LR）
                bb_all = list(_model._bb_params())
                for _m in [_model.se_l1, _model.se_l2, _model.se_l3, _model.se_l4,
                            _model.bf_expand, _model.fusion, _model.proj_cls]:
                    bb_all += list(_m.parameters())
                optimizer = torch.optim.AdamW([
                    {'params': bb_all,
                     'lr': LR_C_BACKBONE,  'name': 'backbone',  'weight_decay': WEIGHT_DECAY},
                    {'params': list(_model.time_head.parameters()),
                     'lr': LR_C_TIME_HEAD, 'name': 'time_head', 'weight_decay': WEIGHT_DECAY},
                    {'params': list(_model.proj_supcon.parameters()),
                     'lr': LR_C_SUPCON,   'name': 'supcon',    'weight_decay': WEIGHT_DECAY},
                ], weight_decay=WEIGHT_DECAY)
                scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
                print(f'\n[E{epoch}] ★ Phase AB→C: drug4_acc={val["drug4_acc"]:.4f}')
                print(f'  drug4_head FROZEN | new optimizer (3 groups) built')
                print(f'  LR: bb={LR_C_BACKBONE:.0e}  th={LR_C_TIME_HEAD:.0e}  sc={LR_C_SUPCON:.0e}\n')

        lr_bb = next((g['lr'] for g in optimizer.param_groups
                      if g.get('name') == 'backbone'), 0.0)
        lr_hd = next((g['lr'] for g in optimizer.param_groups
                      if g.get('name') == 'heads'), optimizer.param_groups[0]['lr'])

        # ── Logging ──────────────────────────────────────────────────────────
        if current_phase == 'AB':
            row = {
                'epoch': epoch, 'phase': current_phase,
                'trn_loss':      round(trn['total'],    4),
                'trn_drug4':     round(trn['drug4'],    4),
                'trn_time':      round(trn['time'],     4),
                'trn_drug4_acc': round(trn['drug4_acc'],4),
                'trn_time_acc':  round(trn['time_acc'], 4),
                'val_loss':      round(val['total'],    4),
                'val_drug4':     round(val['drug4'],    4),
                'val_time':      round(val['time'],     4),
                'val_drug4_acc': round(val['drug4_acc'],4),
                'val_time_acc':  round(val['time_acc'], 4),
                'lr_backbone': f'{lr_bb:.2e}', 'lr_head': f'{lr_hd:.2e}',
                'elapsed_s':   round(elapsed, 1),
                'timestamp':   datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            }
            print(f'{epoch:>4d} [AB]  '
                  f'{trn["total"]:>8.4f} {trn["drug4_acc"]:>7.4f} {trn["time_acc"]:>8.4f} '
                  f'{val["total"]:>8.4f} {val["drug4_acc"]:>7.4f} {val["time_acc"]:>8.4f} '
                  f'{lr_bb:>8.1e} {lr_hd:>8.1e} {elapsed:>5.0f}')
        else:  # Phase C
            lr_sc = next((g['lr'] for g in optimizer.param_groups
                          if g.get('name') == 'supcon'), 0.0)
            lr_th = next((g['lr'] for g in optimizer.param_groups
                          if g.get('name') == 'time_head'), 0.0)
            row = {
                'epoch': epoch, 'phase': current_phase,
                'trn_loss':     round(trn['total'],          4),
                'trn_l1':       round(trn.get('l1_ordinal', 0), 4),
                'trn_l2':       round(trn.get('l2_emd',     0), 4),
                'trn_l3':       round(trn.get('l3_triplet', 0), 4),
                'trn_l4':       round(trn.get('l4_supcon',  0), 4),
                'trn_l5':       round(trn.get('l5_repulse', 0), 4),
                'trn_l6':       round(trn.get('l6_smooth',  0), 4),
                'trn_time_acc': round(trn.get('time_acc',   0), 4),
                'trn_drug4_acc':round(trn.get('drug4_acc',  0), 4),
                'gnorm_supcon': round(trn.get('gnorm_supcon',   0), 4),
                'gnorm_time_h': round(trn.get('gnorm_time_head',0), 4),
                'gnorm_bb':     round(trn.get('gnorm_backbone', 0), 4),
                'val_loss':           round(val['total'],            4),
                'val_drug4_acc':      round(val.get('drug4_acc', 0), 4),
                'val_time_acc':       round(val.get('time_acc',  0), 4),
                'val_kendall_tau':    round(val.get('kendall_tau',0), 4),
                'val_temporal_ord':   round(val.get('temporal_ord_acc',0), 4),
                'smooth_delta':       round(val.get('smooth_delta', SMOOTH_DELTA_INIT), 4),
                'lr_backbone':  f'{lr_bb:.2e}',
                'lr_supcon':    f'{lr_sc:.2e}',
                'lr_time_head': f'{lr_th:.2e}',
                'elapsed_s':    round(elapsed, 1),
                'timestamp':    datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            }
            print(f'{epoch:>4d} [C]  '
                  f'tot={trn["total"]:.4f}  l4={trn.get("l4_supcon",0):.4f}  '
                  f'dr4={val["drug4_acc"]:.4f}  time={val["time_acc"]:.4f}  '
                  f'τ={val.get("kendall_tau",0):.4f}  ord={val.get("temporal_ord_acc",0):.4f}  '
                  f'{elapsed:.0f}s')

        log_rows.append(row)
        pd.DataFrame([row]).to_csv(MT_LOG, mode='a', header=False, index=False)

        # ── Checkpoint & Early Stopping ──────────────────────────────────────
        if current_phase == 'AB':
            monitor = val['drug4_acc']
            if monitor > best_val_acc:
                best_val_acc = monitor
                torch.save({
                    'epoch': epoch, 'model': _model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'val_drug4_acc': val['drug4_acc'],
                    'val_time_acc':  val['time_acc'],
                    'val_loss':      val['total'],
                    'phase': current_phase,
                    'config': {
                        'backbone': BACKBONE, 'embedding_dim': EMBEDDING_DIM,
                        'n_drug': N_DRUG, 'n_time': N_TIME,
                        'feat_dim': FEAT_DIM, 'arch': 'dual_projection_v9',
                        'dropout': DROPOUT, 'drop_path_rate': DROPPATH_RATE},
                }, MT_CKPT)
                print(f'  ✅ 新最优 val_drug4={best_val_acc:.4f}  '
                      f'time={val["time_acc"]:.4f}  → {MT_CKPT}')
        else:  # Phase C — 监控 Kendall tau
            tau_now = val.get('kendall_tau', -1.0)
            if tau_now > best_tau:
                best_tau = tau_now; pat_c = 0
                torch.save({
                    'epoch': epoch, 'model': _model.state_dict(),
                    'optimizer': optimizer.state_dict(),
                    'val_drug4_acc':       val.get('drug4_acc', 0),
                    'val_kendall_tau':      best_tau,
                    'val_temporal_ord_acc': val.get('temporal_ord_acc', 0),
                    'val_time_acc':         val.get('time_acc', 0),
                    'phase': 'C',
                    'config': {
                        'backbone': BACKBONE, 'embedding_dim': EMBEDDING_DIM,
                        'n_drug': N_DRUG, 'n_time': N_TIME,
                        'feat_dim': FEAT_DIM, 'arch': 'dual_projection_v9_phaseC_v12',
                        'dropout': DROPOUT, 'drop_path_rate': DROPPATH_RATE},
                }, MT_CKPT)
                print(f'  ✅ [BEST C] τ={best_tau:.4f}  '
                      f'ord={val.get("temporal_ord_acc",0):.4f}  '
                      f'dr4={val.get("drug4_acc",0):.4f}  → {MT_CKPT}')
            else:
                pat_c += 1
                if pat_c >= PATIENCE_C:
                    print(f'\n  ⏹ Phase C early stop @ epoch {epoch}  best_tau={best_tau:.4f}')
                    break

            # Drug4 guardian：防止药物分类能力在 Phase C 中崩溃
            if val.get('drug4_acc', 1.0) < DRUG4_ACC_GUARDIAN:
                guard_c += 1
                print(f'  [GUARDIAN] drug4_acc={val.get("drug4_acc",0):.4f}'
                      f'<{DRUG4_ACC_GUARDIAN}  count={guard_c}/2')
                if guard_c >= 2:
                    print(f'\n  ⏹ drug4_acc guardian → Phase C 强制停止')
                    break
            else:
                guard_c = 0

        torch.save({
            'epoch': epoch, 'model': _model.state_dict(),
            'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(), 'best_val_acc': best_val_acc,
            'log_rows': log_rows, 'warmup_done': warmup_done,
            'phase': current_phase, 'phase_consec': phase_consec,
            # Phase C 状态
            'best_tau': best_tau, 'pat_c': pat_c, 'guard_c': guard_c,
        }, MT_CKPT_LAST)
        gc.collect(); torch.cuda.empty_cache()

    print(f'\n✅ 训练完成  best_val_drug4_acc={best_val_acc:.4f}  best_tau={best_tau:.4f}')
    print(f'   best  → {MT_CKPT}')
    print(f'   log   → {MT_LOG}')
    return MT_CKPT


MT_CKPT = train_v9(resume=True)


  v9 两阶段训练: RESNET18
  Phase AB: drug4直接四分类 → drug4≥0.95×2 or ep>25
  Phase C:  +SupCon时间结构化 → EarlyStopping(patience=10)
  BATCH=96  LR_bb=5.0e-05  LR_hd=2.0e-04
  ★ Drug4Weight=[1.0, 1.0, 1.0, 1.0]  W_TIME=0.4
[v9] 应用 DropPath (rate=0.1, 线性调度) ...
  block [0] 0 → DropPath(rate=0.0125)
  block [1] 1 → DropPath(rate=0.0250)
  block [2] 0 → DropPath(rate=0.0375)
  block [3] 1 → DropPath(rate=0.0500)
  block [4] 0 → DropPath(rate=0.0625)
  block [5] 1 → DropPath(rate=0.0750)
  block [6] 0 → DropPath(rate=0.0875)
  block [7] 1 → DropPath(rate=0.1000)
[Model] Backbone frozen

开始训练: epoch 1 → 20

  Ep   Ph  TrnLoss  TrnDr4  TrnTime  ValLoss  ValDr4  ValTime    LR_bb    LR_hd     s
------------------------------------------------------------------------------------------


   1 [AB]    1.5955  0.5919   0.2182   2.4870  0.2405   0.2039  0.0e+00  2.0e-04   870
  ✅ 新最优 val_drug4=0.2405  time=0.2039  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth


   2 [AB]    1.3671  0.6949   0.2338   2.4710  0.2702   0.2055  0.0e+00  2.0e-04   872
  ✅ 新最优 val_drug4=0.2702  time=0.2055  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth


   3 [AB]    1.2775  0.7326   0.2421   2.2566  0.2784   0.2002  0.0e+00  2.0e-04   950
  ✅ 新最优 val_drug4=0.2784  time=0.2002  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth
[Model] Backbone unfrozen

[E4] Warmup → backbone 解冻  trainable=12,223,948  groups=4



   4 [AB]    1.0635  0.8288   0.2635   2.3150  0.3011   0.2178  5.0e-05  2.0e-04   981
  ✅ 新最优 val_drug4=0.3011  time=0.2178  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth


...

  16 [AB]    0.5433  0.9763   0.4513   2.4875  0.5274   0.2354  1.3e-05  5.0e-05   901


  17 [AB]    0.5223  0.9808   0.4630   2.7766  0.4812   0.2390  1.3e-05  5.0e-05   915


  18 [AB]    0.5201  0.9817   0.4648   1.8572  0.6394   0.2364  1.3e-05  5.0e-05   942
  ✅ 新最优 val_drug4=0.6394  time=0.2364  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth


  19 [AB]    0.5107  0.9827   0.4690   2.6704  0.5147   0.2372  1.3e-05  5.0e-05   938


  20 [AB]    0.5095  0.9840   0.4742   2.5969  0.5266   0.2643  6.3e-06  2.5e-05   938

✅ 训练完成  best_val_drug4_acc=0.6394  best_tau=-1.0000
   best  → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/best_mt_v9.pth
   log   → /content/drive/MyDrive/drug_effect/pipeline_c/multitask/train_log_v9.csv


## Step 6b — 训练曲线 v9

In [ ]:
# import pandas as pd, matplotlib.pyplot as plt, matplotlib.ticker as mticker
# import numpy as np, os

# if not os.path.exists(MT_LOG):
#     print(f'⚠  日志不存在: {MT_LOG}')
# else:
#     df = pd.read_csv(MT_LOG); eps = df['epoch'].values
#     best_idx   = df['val_drug4_acc'].idxmax()
#     best_epoch = int(df.loc[best_idx, 'epoch'])
#     best_val   = float(df.loc[best_idx, 'val_drug4_acc'])

#     phase_changes = {}
#     cur = df['phase'].iloc[0]
#     for i, ph in enumerate(df['phase']):
#         if ph != cur: phase_changes[ph] = int(df['epoch'].iloc[i]); cur = ph

#     lr_col  = 'lr_backbone' if 'lr_backbone' in df.columns else 'lr'
#     lr_vals = pd.to_numeric(df[lr_col], errors='coerce').values
#     lr_drops = [int(eps[i]) for i in range(1, len(lr_vals))
#                 if pd.notna(lr_vals[i]) and pd.notna(lr_vals[i-1])
#                 and lr_vals[i] < lr_vals[i-1] * 0.5]

#     C_TRN, C_VAL = '#4472C4', '#ED7D31'
#     phase_colors = {'C': '#4CAF50'}
#     phase_bg     = {'AB': '#E3F2FD', 'C': '#E8F5E9'}

#     fig, axes = plt.subplots(2, 3, figsize=(20, 10))

#     def _mark(ax):
#         for ph, ep in phase_changes.items():
#             ax.axvline(ep-0.5, color=phase_colors.get(ph,'gray'),
#                        lw=1.5, ls='--', alpha=0.8, label=f'Phase {ph}')
#         for ep in lr_drops: ax.axvline(ep, color='red', lw=1, ls=':', alpha=0.5)
#         ax.axvline(best_epoch, color='gold', lw=1.5, ls='--', alpha=0.8)

#     def _shade(ax):
#         for _, row in df.iterrows():
#             bg = phase_bg.get(row['phase'], 'white')
#             ax.axvspan(row['epoch']-.5, row['epoch']+.5, alpha=0.12, color=bg, linewidth=0)

#     ax = axes[0,0]
#     ax.plot(eps, df['trn_loss'], C_TRN, lw=2, label='Train')
#     ax.plot(eps, df['val_loss'], C_VAL, lw=2, label='Val')
#     _mark(ax); _shade(ax); ax.set_title('Total Loss'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

#     ax = axes[0,1]
#     ax.plot(eps, df['trn_drug4_acc'], C_TRN, lw=2, label='Train Drug4')
#     ax.plot(eps, df['val_drug4_acc'], C_VAL, lw=2, label='Val Drug4')
#     ax.scatter([best_epoch],[best_val], color='gold', s=100, zorder=5,
#                label=f'Best={best_val:.4f}@{best_epoch}')
#     ax.axhline(PHASE_AB_THRESH, color='gray', lw=1, ls=':', alpha=0.7, label=f'Thresh={PHASE_AB_THRESH}')
#     _mark(ax); _shade(ax); ax.set_title('Drug4 Accuracy ★ v9'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

#     ax = axes[0,2]
#     ax.plot(eps, df['trn_time_acc'], '#9C27B0', lw=2, label='Train Time')
#     ax.plot(eps, df['val_time_acc'], '#FF9800',  lw=2, label='Val Time')
#     _mark(ax); _shade(ax); ax.set_title('Time Accuracy'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

#     ax = axes[1,0]
#     c_df = df[df['phase']=='C']
#     if len(c_df) > 0:
#         ax.plot(c_df['epoch'], c_df['val_kendall_tau'], '#4CAF50', lw=2, label='Kendall τ')
#         ax.plot(c_df['epoch'], c_df['val_temporal_ord'], '#F44336', lw=2, label='Temporal Ord')
#     _mark(ax); ax.set_title('Phase C Temporal Metrics'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

#     ax = axes[1,1]
#     if len(c_df) > 0:
#         ax.plot(c_df['epoch'], c_df['trn_l4'], C_TRN, lw=2, label='Train L4 SupCon')
#         ax.plot(c_df['epoch'], c_df['trn_l1'], '#9C27B0', lw=2, label='Train L1 Ordinal')
#     ax.set_title('Phase C Loss Components'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

#     ax = axes[1,2]
#     ax.plot(eps, lr_vals, '#70AD47', lw=2, marker='o', markersize=3)
#     ax.set_yscale('log')
#     ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1e'))
#     ax.set_title('Learning Rate'); ax.grid(alpha=0.3, which='both')

#     from matplotlib.patches import Patch
#     fig.legend(handles=[Patch(facecolor='#E3F2FD', label='Phase AB: direct drug4 + time'),
#                         Patch(facecolor='#E8F5E9', label='Phase C: +SupCon temporal')],
#                loc='lower center', ncol=2, fontsize=9, bbox_to_anchor=(0.5, -0.02))
#     fig.suptitle(f'v9 训练曲线 — {BACKBONE.upper()}  Best={best_val*100:.2f}% @ep{best_epoch}',
#                  fontsize=13, fontweight='bold')
#     plt.tight_layout()
#     curve_path = os.path.join(MT_DIR, 'training_curves_v9.png')
#     plt.savefig(curve_path, dpi=150, bbox_inches='tight'); plt.show()
#     print(f'✅ → {curve_path}')
#     if phase_changes: print(f'   Phase 转换: {phase_changes}')
